In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import copy
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from typing import Literal

from datasets import load_dataset

pd.set_option("display.max_columns", 150)

root = "ADS599-Capstone/modeling_data"

df_patient = load_dataset(path=root, name='full_patient_state', split='full_patient_state').to_pandas()
df_cohort = pd.read_parquet("data/cohort_df_with_triage_label_corrected.parquet")


torch.manual_seed(10)
np.random.seed(10)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

README.md: 0.00B [00:00, ?B/s]

Generating full_patient_state split:   0%|          | 0/6551723 [00:00<?, ? examples/s]

Using cpu device


In [7]:
df_patient.value_counts("ecg_status")

ecg_status
0.0    3658017
3.0    1402979
2.0    1111426
1.0     379301
Name: count, dtype: int64

In [5]:
df_patient.head()

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,Other,Analgesic - Opioid/NSAID,Antiemetic,Analgesic - Acetaminophen,Antibiotic,Benzodiazepine - Sedative/Anxiolytic,Analgesic - NSAID,Bronchodilator,Antiplatelet,GI - Acid Suppression,Corticosteroid,Beta Blocker,Diuretic,Antipsychotic,Anticoagulant,Insulin/Glucose,Calcium Channel Blocker,Nitrate,ACE Inhibitor,IV Fluid,Anticonvulsant,Antiarrhythmic,weight,ecg_status,rad_status,in_ed,in_ward,ed_boarding,terminal_event,gender,anchor_age,arrival_transport,acuity,height,recon_ace_inhibitor,recon_analgesic___nsaid,recon_analgesic___opioid_nsaid,recon_antiarrhythmic,recon_antibiotic,recon_anticoagulant,recon_anticonvulsant,recon_antiemetic,recon_antiplatelet,recon_antipsychotic,recon_benzodiazepine___sedative_anxiolytic,recon_beta_blocker,recon_bronchodilator,recon_calcium_channel_blocker,recon_corticosteroid,recon_diuretic,recon_gi___acid_suppression,recon_insulin_glucose,recon_nitrate,admission_type,current_temperature,current_heartrate,current_resprate,current_o2sat,current_sbp,current_dbp,current_pain,current_pulse_pressure,current_map,time_since_last_hrs,temperature_rolling2h,temperature_delta,temperature_rate,heartrate_rolling2h,heartrate_delta,heartrate_rate,resprate_rolling2h,resprate_delta,resprate_rate,o2sat_rolling2h,o2sat_delta,o2sat_rate,sbp_rolling2h,sbp_delta,sbp_rate,dbp_rolling2h,dbp_delta,dbp_rate,observe,dispense_meds,ward_transfer,discharge,transfer_icu,ecg_ordered,rad_ordered,vitals_checked,labs_ordered,micro_ordered
0,30000012,11714491,21562392.0,2126-02-14 20:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,0,0,1.0,0,0
1,30000012,11714491,21562392.0,2126-02-14 20:52:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.5,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,1,0,0.0,0,0
2,30000012,11714491,21562392.0,2126-02-14 21:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,1.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,1,0,0,0,0,0,0,0.0,0,0
3,30000012,11714491,21562392.0,2126-02-14 21:52:00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [105]:
dead_ed_stays = df_cohort[df_cohort['discharge_location'] == 'DIED'][['ed_stay_id', 'discharge_location']]

df_patient = df_patient.merge(dead_ed_stays, how='left', on='ed_stay_id')

df_patient['hospital_mortality'] = df_patient['discharge_location'].apply(lambda x: 1 if pd.notna(x) else 0)

df_patient.drop(columns='discharge_location', inplace=True)

Adding an indicator for in hospital mortality status to be the penalty for the trad DQN set up

# Drop Duplicates

In [106]:
drop_cols = df_patient.columns[4:].tolist()
drop_cols.remove("time_since_last_hrs")
drop_cols.append("ed_stay_id")
df_model = df_patient.drop_duplicates(subset=drop_cols)

In [108]:
df_model.drop(columns='ward_transfer', inplace=True)

Below are the steps to instead merge the observe steps.  That seems to drop the last row in some time steps.  By dropping dups, the only ed_stays that are 1 step are the ones that were already one step.  Seems to better preserve the true trajectory of the patient state without changing the distribution much

In [5]:
# Distribution of actions
non_terminal_actions = ['observe', "vitals_checked", "labs_ordered", "micro_ordered", "ecg_ordered", "rad_ordered", "dispense_meds"]
(df_model[non_terminal_actions].sum() / len(df_model)) * 100

observe           36.512308
vitals_checked    30.647173
labs_ordered      19.006599
micro_ordered      2.938714
ecg_ordered        2.882241
rad_ordered        4.783427
dispense_meds      9.074277
dtype: float64

# Remove Consecutive Observe Steps

In [20]:
# Keep only timesteps where something changed drop consecutive observe rows
# This reframes the problem around state changes rather than every clock tick
df_model = df_patient.copy()

# Mark any_action: 1 if any non-observe action occurred at this timestep
non_observe_actions = ["vitals_checked", "labs_ordered", "micro_ordered",
                       "ecg_ordered", "rad_ordered", "dispense_meds", "ward_transfer"]
df_model["any_action"] = df_model[non_observe_actions].any(axis=1).astype(int)

# Flag consecutive observe rows: observe=1 and previous row in same stay was also observe
df_model = df_model.sort_values(["ed_stay_id", "time"]).reset_index(drop=True)
prev_any = df_model.groupby("ed_stay_id")["any_action"].shift(1).fillna(0)
is_consec_observe = (df_model["any_action"] == 0) & (prev_any == 0)

# Keep: rows with any action, plus the first observe row in each consecutive observe block
df_model = df_model[~is_consec_observe].copy()

# Drop ward_transfer â€” treating escalation/discharge as terminal reward, not action
df_model = df_model.drop(columns=["ward_transfer"])

df_model = df_model.drop(columns=["any_action"]).reset_index(drop=True)

print(f"Original rows: {len(df_patient):,}")
print(f"Filtered rows: {len(df_model):,}")
print(f"Reduction:     {1 - len(df_model)/len(df_patient):.1%}")
print(f"Action rates after filtering:")
action_check = ["observe", "vitals_checked", "labs_ordered", "micro_ordered",
                "ecg_ordered", "rad_ordered", "dispense_meds"]
for a in action_check:
    print(f"  {a:<20} {df_model[a].mean():.4f}")

Original rows: 6,551,723
Filtered rows: 1,237,071
Reduction:     81.1%
Action rates after filtering:
  observe              0.3828
  vitals_checked       0.2782
  labs_ordered         0.2531
  micro_ordered        0.0292
  ecg_ordered          0.0262
  rad_ordered          0.0434
  dispense_meds        0.0824


In [21]:
# Remove ed stays that are only one time step
short_frames = df_model.groupby('ed_stay_id').size()
ed_removed = short_frames[short_frames == 1].index
short_frames = short_frames[short_frames > 1]
df_model = df_model[df_model['ed_stay_id'].isin(short_frames.index)]

print(ed_removed)

Index([31128182, 31176205, 33124406, 33534225, 35193499, 39377306, 39387127], dtype='int64', name='ed_stay_id')


In [34]:
deduped_patient.groupby('ed_stay_id').size().sort_values().head(10)

ed_stay_id
39387127    1
39377306    1
31128182    1
33124406    1
35193499    1
31176205    1
33534225    1
37455800    2
36799584    2
39570863    2
dtype: int64

# Preprocess Data

In [ ]:
# Map Gender
gender_map = {"F": 1, "M": 0}
df_model["gender"] = df_model["gender"].map(gender_map)

# Change acuity to integer
df_model["acuity"] = df_model["acuity"].astype(int)

# Mask height and weight
df_model["height_missing"] = df_model["height"].isna().astype(int)
df_model["weight_missing"] = df_model["weight"].isna().astype(int)
df_model[["height", "weight"]] = df_model[["height", "weight"]].fillna(0)

# Create pain_missing column and convert the Other category to 0
df_model["pain_missing"] = (df_model["current_pain"] == "Other").astype(int)
df_model["current_pain"] = pd.to_numeric(df_model["current_pain"], errors="coerce").fillna(0)

# Mask admission type then one hot encode admission and arrival
df_model["admission_missing"] = df_model["admission_type"].isna().astype(int)
at_dummies = pd.get_dummies(df_model["admission_type"], prefix="admission_type", dummy_na=False, dtype=int)
arrival_dummies = pd.get_dummies(df_model["arrival_transport"], prefix="arrival_transport", dtype=int)
df_model_updated = pd.concat([df_model, at_dummies, arrival_dummies], axis=1).drop(columns=["admission_type", "arrival_transport"])

In [114]:
# Columns out of order so this pieces all the state cols together
vitals = [c for c in df_model_updated.loc[:, "current_temperature":"dbp_rate"].columns if not c.endswith("_rate") and not c.endswith("_delta")]
med_cols = [c for c in df_model_updated.columns if c.startswith("recon")]
admission_cols = at_dummies.columns.to_list() + ["admission_missing"]
arrival_cols = arrival_dummies.columns.to_list()

# Full State
state = df_model_updated.columns[4:71].to_list() + ["gender", "anchor_age", "acuity", "height", "height_missing", "weight_missing", "pain_missing"] + med_cols + vitals + admission_cols + arrival_cols

# Action columns
actions = ['observe', "vitals_checked", "labs_ordered", "micro_ordered", "ecg_ordered", "rad_ordered", "dispense_meds", "discharge", "transfer_icu"]

# Terminal Columns
terminal_cols = ["discharge", "transfer_icu"]

# Penalty Indicator
hospital_mortality = "hospital_mortality"

# Terminal Label for patients
terminal_label = "terminal_event"

# RL Setup

- traditional DQN is learning to approximate the Q function so the agent never actually takes an action.  Can only penalize or reward whatever is present in your data.  So might swap out the improper transfer for death instead

In [425]:
# Reproducibility
random_seed = 10

# For reinforcement learning process
discount_factor = 0.9 # discounts future rewards

# DQN variables
hidden_states = [256, 256, 128]
input_size = len(state)
action_space = 9
dropout_rate = 0.1
activation_function = nn.ReLU()
loss_fn = nn.MSELoss()
batch_size = 16
optimizer_function = torch.optim.AdamW
learning_rate = 1e-6
update_steps = 500
epochs = 5
cql_alpha = 0.1

# Data Specific
train_size = 0.2
test_size = 0.1

**Bellman eq**
v(state) = max_action [R(s, a) + discount_factor * v(next_state)]

# Split Data and Scale

In [115]:
stay_actions = (
    df_model_updated.groupby("ed_stay_id")[actions]
    .max()
    .reset_index()
)

# Train
train_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=train_size, random_state=random_seed)
resample_idx, sample_idx = next(train_split.split(stay_actions, stay_actions[actions]))
sample_stays = set(stay_actions.iloc[sample_idx]['ed_stay_id'])
df_train = df_model_updated[df_model_updated['ed_stay_id'].isin(sample_stays)].copy()

# Test
stay_test = stay_actions.loc[resample_idx]
test_split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_seed)
_, sample_idx = next(test_split.split(stay_test, stay_test[actions]))
sample_stays = set(stay_test.iloc[sample_idx]['ed_stay_id'])
df_test = df_model_updated[df_model_updated['ed_stay_id'].isin(sample_stays)].copy()

print(f"Sample stays: {df_train['ed_stay_id'].nunique():,}")
print(df_train.drop_duplicates('ed_stay_id')['terminal_event'].value_counts())
print(f"\nSample stays: {df_test['ed_stay_id'].nunique():,}")
print(df_test.drop_duplicates('ed_stay_id')['terminal_event'].value_counts())

train_idx = set(df_train.index)
test_idx = set(df_test.index)

print(f"\nOverlap: {train_idx.intersection(test_idx)}")

Sample stays: 16,843
terminal_event
DISCHARGE_ED      7580
DISCHARGE_WARD    5613
ICU               3650
Name: count, dtype: int64

Sample stays: 6,737
terminal_event
DISCHARGE_ED      3068
DISCHARGE_WARD    2209
ICU               1460
Name: count, dtype: int64

Overlap: set()


In [117]:
scale = StandardScaler()
scaling_cols = vitals + ['anchor_age', 'weight', 'height', 'time_since_last_hrs']

df_train[scaling_cols] = scale.fit_transform(df_train[scaling_cols])
df_test[scaling_cols] = scale.transform(df_test[scaling_cols])

In [119]:
# Code terminal event
terminal_event_mapping = {"DISCHARGE_WARD": 0, "DISCHARGE_ED": 0, "ICU": 1}
df_train['terminal_code'] = df_train['terminal_event'].map(terminal_event_mapping)
df_test['terminal_code'] = df_test['terminal_event'].map(terminal_event_mapping)

In [120]:
# Map actions to single column for conversion
# Making discharge and transfer_icu here a higher int because they aren't exclusive in the data.  Lot of overlap with vitals and labs in particular.  another problem for another day
action_mapping = {'observe': 0, "vitals_checked": 1, "dispense_meds": 2, "labs_ordered": 3, "micro_ordered": 4, "ecg_ordered": 5, "rad_ordered": 6, "discharge": 7, "transfer_icu": 8}

for a in actions:
    df_train[a] = df_train[a].apply(lambda x: action_mapping[a] if x == 1 else np.nan)
    df_test[a] = df_test[a].apply(lambda x: action_mapping[a] if x == 1 else np.nan)

# This is used for single action reinforcement learning
df_train['action_taken'] = df_train[actions].max(axis=1)
df_test['action_taken'] = df_test[actions].max(axis=1)

In [122]:
# The nan's are the rows where 'ward_transfer' would have been
df_train.dropna(subset='action_taken', inplace=True)
df_test.dropna(subset='action_taken', inplace=True)

In [124]:
# Add a time step column to calculate penalty for time
df_train['time_steps'] = df_train.sort_values(by='time').groupby('ed_stay_id').cumcount()
df_test['time_steps'] = df_test.sort_values(by='time').groupby('ed_stay_id').cumcount()

In [125]:
# Code a terminal state column so we know in the buffer when the terminal state actually is
df_train['terminal_state'] = df_train.groupby('ed_stay_id')['time_steps'].transform(lambda x: x == (x.count()-1))
df_test['terminal_state'] = df_test.groupby('ed_stay_id')['time_steps'].transform(lambda x: x == (x.count()-1))

In [126]:
missing = [c for c in state if c not in df_train.columns]
# extra = [c for c in df_train.columns if c not in state + actions + ['ed_stay_id', 'subject_id', 'hadm_id', 'time', 'ed_boarding', 'terminal_event']]
print(f"In state_cols but not in df_train: {missing}")
# print(f"In df_train but not accounted for: {extra}")

In state_cols but not in df_train: []


In [ ]:
# df_train.drop(columns=extra, inplace=True)
# df_test.drop(columns=extra, inplace=True)

# Convert Data to Tensors

In [16]:
from torch.utils.data import RandomSampler, BatchSampler

In [128]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

df_train.sort_values(by=['ed_stay_id', 'time'], inplace=True)
df_test.sort_values(by=['ed_stay_id', 'time'], inplace=True)

grouped_train = df_train.groupby('ed_stay_id', as_index=False)
grouped_test = df_test.groupby('ed_stay_id', as_index=False)

# MDP tuple (s, r, a, t, ns)
# reward function is tunable so instead of calculating here, just getting the raw inputs for the function
state_train = torch.tensor(df_train[state].values.astype(np.float32))
time_train = torch.tensor(df_train['time_steps'].values.astype(np.float32))
action_train = torch.tensor(df_train["action_taken"].values.astype(np.int32))
terminal_train = torch.tensor(df_train[['terminal_code', 'terminal_state', 'hospital_mortality']].values.astype(np.float32))
next_state_train = torch.tensor(grouped_train[state].shift(-1).fillna(0).values.astype(np.float32)) # we fill the nan terminal row with all 0's, this is why we have the terminal flag

state_test  = torch.tensor(df_test[state].values.astype(np.float32))
time_test = torch.tensor(df_test['time_steps'].values.astype(np.float32))
action_test  = torch.tensor(df_test['action_taken'].values.astype(np.int32))
terminal_test = torch.tensor(df_test[['terminal_code', 'terminal_state', 'hospital_mortality']].values.astype(np.float32))
next_state_test = torch.tensor(grouped_test[state].shift(-1).fillna(0).values.astype(np.float32)) 

final_train = TensorDataset(state_train, time_train, action_train, terminal_train, next_state_train)
r_sampler = RandomSampler(final_train, replacement=False)
batch_sampler = BatchSampler(r_sampler, batch_size=batch_size, drop_last=True)

train_loader = DataLoader(
    dataset=final_train,
    shuffle=False,
    batch_sampler=batch_sampler
)
test_loader = DataLoader(
    TensorDataset(state_test, time_test, action_test, terminal_test, next_state_test),
    batch_size=batch_size,
    shuffle=False,
)

print(f"State dim: {state_train.shape}")
print(f"Action dim: {action_train.shape}")
print(f"Terminal dim: {terminal_train.shape}")
print(f"Time steps dim: {time_train.shape}")
print(f"Next State dim: {next_state_train.shape}")

State dim: torch.Size([219954, 124])
Action dim: torch.Size([219954])
Terminal dim: torch.Size([219954, 3])
Time steps dim: torch.Size([219954])
Next State dim: torch.Size([219954, 124])


In [133]:
# Column Mapping
terminal_label_mapping = {0: 'DISCHARGE_WARD', 0: "DISCHARGE_ED", 1: "ICU"}
terminal_code_mapping = {0: 'discharge', 1: 'transfer_icu'}
terminal_state_mapping = {0: 'No', 1: "Yes"}
state_feature_mapping = dict(zip(range(0, len(state)), state))
action_reverse_mapping = {v: k for k, v in action_mapping.items()}

# Replay Buffer

In [19]:
# (s, a, r, s', terminal)
index_mapping = {0: "state", 1: "time_steps", 2: "action", 3: "terminal", 4: "next_state"}
tuple_mapping = {v: k for k, v in index_mapping.items()}

In [468]:
terminal_imbalance = df_test[df_test['terminal_state'] == 1].value_counts('terminal_code', normalize=True)

In [470]:
class_ratio = terminal_imbalance[0] / terminal_imbalance[1]

In [487]:
def reward_function(action, terminal_code, timestep: int, class_ratio=2, time_weight: float=-0.001, reward: float=0.4, penalty: float=0.0):
    action_label = action_reverse_mapping[action.item()]
    terminal_label = terminal_code_mapping[terminal_code[0].item()]
    is_terminal = terminal_code[1].item()
    hospital_mortality = terminal_code[2].item()
    lt_reward = 0
    time_penalty = timestep * time_weight

    if is_terminal:
        if action_label == 'discharge' and action_label == terminal_label:
            lt_reward += reward
        elif action_label == 'transfer_icu' and action_label == terminal_label:
            lt_reward += (reward * class_ratio)
        if hospital_mortality == 1:
            lt_reward -= penalty
        
    return lt_reward + time_penalty

batch = next(iter(train_loader))

actions = batch[tuple_mapping['action']]
terminal_states = batch[tuple_mapping['terminal']]
time_steps = batch[tuple_mapping['time_steps']]

rewards = [reward_function(actions[i], terminal_states[i], time_steps[i]) for i in range(batch_size)]

# Network

In [488]:
class ProviderNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        self.input_size = input_size # -> maybe make this a model arg
        self.output_size = action_space # -> maybe make this an input
        self.activation_function = activation_function # -> model arg
        self.dropout_rate = dropout_rate # -> model arg

        in_state = self.input_size
        for h in hidden_states:
            layer = nn.Linear(in_state, h)
            layers.append(layer)
            layers.append(self.activation_function)
            if self.dropout_rate > 0:
                layers.append(nn.Dropout(p=self.dropout_rate))
            in_state = h
        

        layers.append(nn.Linear(in_state, self.output_size))

        self.linear_activation = nn.Sequential(*layers)
    
    def forward(self, x):
        q_values = self.linear_activation(x)
        return q_values

train_network = ProviderNetwork()
target_network = copy.deepcopy(train_network)
print(train_network)

ProviderNetwork(
  (activation_function): ReLU()
  (linear_activation): Sequential(
    (0): Linear(in_features=124, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=128, out_features=9, bias=True)
  )
)


# Training Loop

In [22]:
tuple_mapping

{'state': 0, 'time_steps': 1, 'action': 2, 'terminal': 3, 'next_state': 4}

In [473]:
optimizer = optimizer_function(train_network.parameters(), lr=learning_rate)

In [348]:
def bellman(model, reward, next_state, terminal, discount_factor=discount_factor):
    terminal_values = terminal[:, 1]
    with torch.no_grad():
        outputs = model(next_state).max(dim=1)
        max_q_values = outputs.values
        max_q_index = outputs.indices
    values = reward + (discount_factor * max_q_values * (1 - terminal_values)) # minus 1 sets this part to 0 if the row is a terminal state
    return values, max_q_index

In [152]:
def update_target_network():
    target_network.load_state_dict(train_network.state_dict())

## Sample

In [136]:
batch = next(iter(train_loader))

In [137]:
s = batch[0]
t = batch[1]
a = batch[2]
terminal = batch[3]
ns = batch[4]

In [138]:
rewards = torch.tensor([reward_function(action=a[i], terminal_code=terminal[i], timestep=t[i]) for i in range(batch_size)])

In [ ]:
q_targets, _ = bellman(model=train_network, reward=rewards, next_state=ns, terminal=terminal)

In [140]:
outputs = train_network(s)

In [200]:
terminal_indices = terminal[:, 1].argwhere().squeeze(1)

In [313]:
q_values = torch.gather(outputs, dim=1, index=a.unsqueeze(1)).squeeze(1)

In [319]:
q_loss = loss_fn(q_values, q_targets)

In [340]:
term = 0.9 * (outputs.mean() - q_values.mean())

In [341]:
term

tensor(-0.0071, grad_fn=<MulBackward0>)

In [335]:
q_values.mean()

tensor(0.0179, grad_fn=<MeanBackward0>)

## Training

In [489]:
def training_loop(dataloader, model, loss_fn, optimizer, bellman_model, alpha=cql_alpha):
    size = len(dataloader.dataset)
    model.train()
    total_q_loss = 0
    for batch, (s, time, action, terminal, ns) in enumerate(dataloader):

        # non_terminal_mask = terminal[:, 1] == 0 # this masks terminal states so the cql penalty is only applied in the case of non-terminal states
        
        optimizer.zero_grad()
        rewards = torch.tensor([reward_function(action=action[i], terminal_code=terminal[i], timestep=time[i]) for i in range(batch_size)])
        q_targets, _ = bellman(model=bellman_model, reward=rewards, next_state=ns, terminal=terminal)
        outputs = model(s)
        q_values = torch.gather(outputs, dim=1, index=action.unsqueeze(1)).squeeze(1)
        q_loss = loss_fn(q_values, q_targets) 

        cql_loss = q_loss + alpha * (outputs.mean() - q_values.mean()) # first term pushes down the estimated actions that are not strongly represented and the second pushes up the Q-values for the actions that were observed in the dataset with that state

        # if non_terminal_mask.any():
        #     cql_loss = q_loss + alpha * (outputs[non_terminal_mask].mean() - q_values[non_terminal_mask].mean())
        # else:
        #     cql_loss = q_loss

        cql_loss.backward()
        optimizer.step()
        total_q_loss += cql_loss.item()

        if (batch + 1) % update_steps == 0:
            update_target_network()

        if batch % 1000 == 0:
            loss, current = q_loss.item(), batch * batch_size + len(s)
            print(f"loss: {loss:>7.4f} [{current:>5d}/{size:>5d}]")
    return total_q_loss / len(dataloader)

In [490]:
def evaluation_loop(dataloader, model, loss_fn, bellman_model):
    num_batches = len(dataloader)
    test_q_loss = 0
    preds = []
    true_terminal = []
    full_preds = []

    model.eval()
    with torch.no_grad():
        for s, time, action, terminal, ns in dataloader:
            
            # Add the true terminal labels
            terminal_indices = terminal[:, 1].argwhere().squeeze(1) # this slices to the terminal state dimension and returns the indices if there is a terminal state
            true_terminal.extend(terminal[terminal_indices, 0]) # get the first column for the true terminal action taken

            rewards = torch.tensor([reward_function(action=action[i], terminal_code=terminal[i], timestep=time[i]) for i in range(len(action))])

            q_targets, _ = bellman(model=bellman_model, reward=rewards, next_state=ns, terminal=terminal)
            outputs = model(s)
            q_values = torch.gather(outputs, dim=1, index=action.unsqueeze(1)).squeeze(1)
            test_q_loss += loss_fn(q_values, q_targets)

            # Get the agent recommendations for terminal states
            preds.extend(outputs[terminal_indices].argmax(dim=1))

            # Add the models predictions for every row
            full_preds.extend(outputs.argmax(dim=1))

    test_q_loss /= num_batches
    # f1_macro = f1(final_preds, true_y).item()
    # f1_per = f1_per_class(final_preds, true_y)
    print(f'\nTest Error: Average Loss: {test_q_loss:>1.4f}')
    # for action, score in zip(action_cols, f1_per.tolist()):
    #     print(f'{action}: {score:.4f}')

    return preds, true_terminal, test_q_loss, full_preds

In [491]:
train_losses = []
eval_losses = []
best = 10
patience = 2
no_patience = 2 # for model getting worse
min_delta = 1e-3
max_delta = 1e-2

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}\n------------------------------")
    t_loss = training_loop(train_loader, train_network, loss_fn, optimizer, target_network)
    train_losses.append(t_loss)

    final_preds, true_terminal, test_q_loss, full_preds = evaluation_loop(test_loader, target_network, loss_fn, train_network)
    eval_losses.append(test_q_loss)

    if test_q_loss < best:
        update_target_network()
        best = test_q_loss
    elif abs(test_q_loss - best) <= min_delta:
        patience -= 1
    elif abs(test_q_loss - best) >= max_delta:
        no_patience -= 1
    
    if not patience or not no_patience:
        break


Epoch 1
------------------------------
loss:  0.0747 [   16/219954]
loss:  0.0271 [16016/219954]
loss:  0.0302 [32016/219954]
loss:  0.0246 [48016/219954]
loss:  0.0357 [64016/219954]
loss:  0.0318 [80016/219954]
loss:  0.0204 [96016/219954]
loss:  0.0329 [112016/219954]
loss:  0.0186 [128016/219954]
loss:  0.0314 [144016/219954]
loss:  0.0346 [160016/219954]
loss:  0.0713 [176016/219954]
loss:  0.0335 [192016/219954]
loss:  0.0724 [208016/219954]

Test Error: Average Loss: 0.0384

Epoch 2
------------------------------
loss:  0.0746 [   16/219954]
loss:  0.0215 [16016/219954]
loss:  0.0213 [32016/219954]
loss:  0.0231 [48016/219954]
loss:  0.0257 [64016/219954]
loss:  0.0228 [80016/219954]
loss:  0.0240 [96016/219954]
loss:  0.0623 [112016/219954]
loss:  0.0667 [128016/219954]
loss:  0.0176 [144016/219954]
loss:  0.0155 [160016/219954]
loss:  0.1043 [176016/219954]
loss:  0.0313 [192016/219954]
loss:  0.0313 [208016/219954]

Test Error: Average Loss: 0.0385

Epoch 3
-----------------

In [ ]:
from sklearn.metrics import f1_score, recall_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Reference mappings — action index to name and terminal outcome encoding
print("Action mapping (name -> index):")
print(action_mapping)
print("\nAction reverse mapping (index -> name):")
print(action_reverse_mapping)
print("\nTerminal code mapping:")
print(terminal_code_mapping)


In [ ]:
# Attach model predictions to the full test dataframe (all timesteps)
df_test['preds'] = full_preds

# Compare predicted vs true action distribution across all timesteps.
# This shows whether the agent is recommending actions at realistic rates
# relative to how often those actions actually occurred in the data.
fp_vals, fp_counts = np.unique(full_preds, return_counts=True)
at_vals, at_counts = np.unique(action_test, return_counts=True)

print("Predicted action distribution (all timesteps):")
for v, c in zip(fp_vals, fp_counts):
    print(f"  {str(action_reverse_mapping.get(v, v)):<25}  {c:>6}  ({c/len(full_preds):.2%})")

print("\nTrue action distribution (all timesteps):")
for v, c in zip(at_vals, at_counts):
    print(f"  {str(action_reverse_mapping.get(v, v)):<25}  {c:>6}  ({c/len(action_test):.2%})")


In [ ]:
# Isolate predictions and true labels at terminal timesteps only.
# These are the final states of each ED stay — where the agent makes
# its disposition recommendation (discharge or transfer to ICU).
preds = np.asarray(final_preds)
true  = np.asarray(true_terminal)

# Distribution of agent action recommendations at terminal states
print("Terminal state — predicted action distribution:")
tp_vals, tp_counts = np.unique(preds, return_counts=True)
for v, c in zip(tp_vals, tp_counts):
    print(f"  {str(action_reverse_mapping.get(int(v), v)):<25}  {c:>6}  ({c/len(preds):.2%})")

# Of all terminal states, what fraction did the agent recommend a terminal action?
# Actions > 6 are terminal (discharge or transfer_icu); anything lower is a non-terminal
# action (observe, order labs, etc.) which would be incorrect at a terminal state.
false = np.where(preds > 6, 1, 0)
terminal_ratio = false.sum() / len(false)
print(f"\nTerminal action rate at terminal states: {terminal_ratio:.2%}")
print("(Remainder were non-terminal actions — observe, labs, etc.)")

# Indices where agent recommended a terminal action — used for disposition evaluation below
terminal_index = np.argwhere(false)


In [ ]:
# Convert terminal action predictions to disposition labels.
# Evaluation is restricted to the subset where the agent actually
# recommended a terminal action (discharge or transfer_icu).
new_preds = np.where(preds[terminal_index] == action_mapping['discharge'], 'discharge', 'transfer_icu')
new_true  = np.where(true[terminal_index] == 0, 'discharge', 'transfer_icu')

# Compare predicted vs true disposition for the terminal-action subset
np_vals, np_counts = np.unique(new_preds, return_counts=True)
nt_vals, nt_counts = np.unique(new_true,  return_counts=True)

print("Predicted disposition (terminal-action subset):")
for v, c in zip(np_vals, np_counts):
    print(f"  {v:<15}  {c:>5}  ({c/len(new_preds):.2%})")

print("\nTrue disposition (terminal-action subset):")
for v, c in zip(nt_vals, nt_counts):
    print(f"  {v:<15}  {c:>5}  ({c/len(new_true):.2%})")


In [ ]:
# Disposition classification metrics — positive class is transfer_icu
# (identifying ICU transfers is the higher-stakes prediction)
f1     = f1_score(new_true, new_preds, pos_label='transfer_icu')
recall = recall_score(new_true, new_preds, pos_label='transfer_icu')
print(f"F1 Score (transfer_icu):  {f1:.4f}")
print(f"Recall   (transfer_icu):  {recall:.4f}")

# Confusion matrix — rows = true disposition, cols = predicted disposition
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(new_true, new_preds, ax=ax,
                                        display_labels=['discharge', 'transfer_icu'])
ax.set_title("Disposition Confusion Matrix\n(terminal-action subset)")
plt.tight_layout()
plt.show()


In [ ]:
import os, json
from datetime import datetime
from sklearn.metrics import confusion_matrix

# ── User notes for this run ───────────────────────────────────────────
run_notes = ""  # <- fill in before running

# ── Config ────────────────────────────────────────────────────────────
config = {
    "hidden_states":        hidden_states,
    "action_space":         action_space,
    "dropout_rate":         dropout_rate,
    "activation_function":  type(activation_function).__name__,
    "loss_fn":              type(loss_fn).__name__,
    "batch_size":           batch_size,
    "optimizer_function":   optimizer_function.__name__,
    "learning_rate":        learning_rate,
    "update_steps":         update_steps,
    "epochs":               epochs,
    "cql_alpha":            cql_alpha,
}

# ── Distributions ─────────────────────────────────────────────────────
fp_vals, fp_counts = np.unique(full_preds, return_counts=True)
fp_dist = {int(v): round(float(c) / len(full_preds), 4) for v, c in zip(fp_vals, fp_counts)}

tp_vals, tp_counts = np.unique(preds, return_counts=True)
tp_dist = {int(v): round(float(c) / len(preds), 4) for v, c in zip(tp_vals, tp_counts)}

np_vals, np_counts = np.unique(new_preds, return_counts=True)
np_dist = {str(v): round(float(c) / len(new_preds), 4) for v, c in zip(np_vals, np_counts)}

nt_vals, nt_counts = np.unique(new_true, return_counts=True)
nt_dist = {str(v): round(float(c) / len(new_true), 4) for v, c in zip(nt_vals, nt_counts)}

# ── Terminal action ratio ──────────────────────────────────────────────
terminal_ratio = float(false.sum() / len(false))

# ── Metrics ───────────────────────────────────────────────────────────
cm = confusion_matrix(new_true.flatten(), new_preds.flatten(), labels=['discharge', 'transfer_icu'])
cm_dict = {
    "labels": ["discharge", "transfer_icu"],
    "matrix": cm.tolist(),
}

# ── Assemble record ───────────────────────────────────────────────────
record = {
    "date":                     datetime.now().strftime("%Y-%m-%d %H:%M"),
    "notes":                    run_notes,
    "config":                   config,
    "best_eval_loss":           round(float(best), 6),
    "full_preds_dist":          fp_dist,
    "final_preds_terminal_dist": tp_dist,
    "terminal_action_ratio":    round(terminal_ratio, 4),
    "new_preds_dist":           np_dist,
    "new_true_dist":            nt_dist,
    "f1_transfer_icu":          round(float(f1), 4),
    "recall_transfer_icu":      round(float(recall), 4),
    "confusion_matrix":         cm_dict,
}

# ── Append to results file ─────────────────────────────────────────────
results_path = "rl_agent_results.jsonl"
with open(results_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(record) + "\n")

print(f"Run logged to {results_path}")
print(f"  Date:                  {record['date']}")
print(f"  Best eval loss:        {record['best_eval_loss']}")
print(f"  Terminal action ratio: {record['terminal_action_ratio']:.2%}")
print(f"  F1 (transfer_icu):     {record['f1_transfer_icu']}")
print(f"  Recall (transfer_icu): {record['recall_transfer_icu']}")
print(f"  Confusion matrix ({cm_dict['labels']}):")
print(f"    {cm.tolist()}")


In [ ]:
results_df = pd.read_json("rl_agent_results.jsonl", lines=True)